In [ ]:
'''
note: for raspberry pi
'''


import cv2
import time

# GStreamer pipeline for IMX219
def gstreamer_pipeline(
    capture_width=1280, capture_height=720, display_width=1280, display_height=720, framerate=30, flip_method=0
):
    return (
        f"nvarguscamerasrc ! video/x-raw(memory:NVMM), width={capture_width}, height={capture_height}, "
        f"format=(string)NV12, framerate=(fraction){framerate}/1 ! nvvidconv flip-method={flip_method} ! "
        f"video/x-raw, width={display_width}, height={display_height}, format=(string)BGRx ! "
        f"videoconvert ! video/x-raw, format=(string)BGR ! appsink drop=true"
    )

cap = cv2.VideoCapture(gstreamer_pipeline(), cv2.CAP_GSTREAMER)

if not cap.isOpened():
    print("Error: Unable to open IMX219 camera.")
    exit()

print("Press 's' to start/stop recording, 'q' to quit.")

recording = False
video_writer = None

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Frame not captured.")
        break

    cv2.imshow("IMX219 Camera Feed", frame)

    if recording and video_writer is not None:
        video_writer.write(frame)

    # Key control:
    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        if not recording:
            filename = f"recorded_{int(time.time())}.avi"
            fourcc = cv2.VideoWriter_fourcc(*'XVID')
            video_writer = cv2.VideoWriter(filename, fourcc, 30.0, (frame.shape[1], frame.shape[0]))
            print(f"Recording started: {filename}")
        else:
            print("Recording stopped.")
            video_writer.release()
            video_writer = None
        recording = not recording

    elif key == ord('q'):
        print("Exiting...")
        break

cap.release()
if video_writer:
    video_writer.release()
cv2.destroyAllWindows()